In [2]:
import asyncio
import re
import pyperclip
from playwright.async_api import async_playwright

SEARCH_URL = "http://localhost:3000"


# =========================
# Normalize URL helper
# =========================
def normalize_url(url: str) -> str:
    url = re.sub(r"\s*›\s*", "/", url)
    url = re.sub(r"/{2,}", "/", url)
    url = url.replace("https:/", "https://")
    return url.strip()


# =========================
# Parse DuckDuckGo clipboard
# =========================
def parse_duckduckgo(raw_text: str):
    lines = [l.strip() for l in raw_text.splitlines() if l.strip()]
    results = []

    i = 0
    while i < len(lines):
        line = lines[i]

        if re.search(r"https?://", line):
            url = normalize_url(line)
            title = None
            content = None

            j = i + 1
            while j < len(lines):
                if not re.search(r"https?://", lines[j]):
                    if title is None:
                        title = lines[j]
                    elif content is None:
                        content = lines[j]
                        break
                j += 1

            if title:
                results.append({
                    "title": title,
                    "url": url,
                    "content": content
                })

            i = j
        else:
            i += 1

    return results


# =========================
# Parse Google clipboard
# =========================
def parse_google(raw_text: str):
    lines = [l.strip() for l in raw_text.splitlines() if l.strip()]
    results = []

    for idx in range(len(lines)):

        line = lines[idx]

        # Google URL line often contains breadcrumb "›"
        if "https://" in line or "›" in line:

            url = normalize_url(line)

            # Title is usually 2 lines above
            title = lines[idx - 2] if idx >= 2 else None

            # Snippet usually after "PDF"
            snippet = None
            if idx + 1 < len(lines) and lines[idx + 1] == "PDF":
                if idx + 2 < len(lines):
                    snippet = lines[idx + 2]
            else:
                if idx + 1 < len(lines):
                    snippet = lines[idx + 1]

            # Only keep pdf-related links
            if url.startswith("https://") and "pdf" in url.lower():
                results.append({
                    "title": title,
                    "url": url,
                    "content": snippet
                })

    return results


# =========================
# Main search engine
# =========================
async def searchEngine(query: str, headless=True, google_engine=False):

    # If Google mode → use DuckDuckGo bang
    if google_engine:
        query = "!g " + query

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        context = await browser.new_context()
        page = await context.new_page()

        # ===== Open search engine =====
        await page.goto(SEARCH_URL)
        await asyncio.sleep(5)

        # ===== Search =====
        print(f"🔍 Searching: {query}")
        await page.keyboard.press("Control+L")
        await page.keyboard.press("Control+A")
        await page.keyboard.press("Backspace")
        await page.keyboard.type(query, delay=20)
        await page.keyboard.press("Enter")

        print("⏳ Waiting for results...")
        await asyncio.sleep(10)
        if google_engine:
            await page.keyboard.press("Control+L")
            await page.keyboard.press("Control+A")
            await page.keyboard.press("Control+C")
            await asyncio.sleep(1)
            raw_link = pyperclip.paste()
            print("raw_link:", raw_link)

            await asyncio.sleep(3)
            await page.keyboard.press("Backspace")

            view_source_query = "view-source:" + raw_link
            await page.keyboard.type(view_source_query, delay=20)
            await page.keyboard.press("Enter")

            print("⏳ Waiting access view-source...")
            await asyncio.sleep(10)
            print("📋 Copying results...")
            await page.keyboard.press("Control+A")
            await page.keyboard.press("Control+C")
            await asyncio.sleep(1)
            raw_text = pyperclip.paste()

            # Save to file
            with open("google_result.html", "w", encoding="utf-8") as f:
                f.write(raw_text)

            print("✅ Saved clipboard content to google_result.html")

            

        else: 

            # ===== Copy results =====
            print("📋 Copying results...")
            await page.keyboard.press("Control+A")
            await page.keyboard.press("Control+C")
            await asyncio.sleep(1)

            raw_text = pyperclip.paste()

        await browser.close()

    print(f"✅ Clipboard captured ({len(raw_text)} chars)")

    # ===== Parse depending on mode =====
    if google_engine:
        print("🌍 Parsing in GOOGLE mode...")
        return parse_google(raw_text)
    else:
        print("🦆 Parsing in DUCKDUCKGO mode...")
        return parse_duckduckgo(raw_text)


# =========================
# Wrapper main
# =========================
async def main(query, headless=True, google_engine=False):
    results = await searchEngine(query, headless, google_engine)
    return results

In [3]:
import os
from langchain_openai import ChatOpenAI

# Configure ProxyPal for ChatGPT
os.environ["OPENAI_API_KEY"] = "proxypal-local"
os.environ["OPENAI_API_BASE"] = "http://localhost:8317/v1"

import os
import json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class SearchReportValidator:
    def __init__(self, model_name="gemini-2.5-flash"):
        self.model_name = model_name
        self.llm = ChatOpenAI(model=model_name, temperature=0.0)

    def run_chatgpt(self, user_prompt: str) -> str:
        prompt = ChatPromptTemplate.from_messages([
            ("user", "{input}")
        ])
        chain = prompt | self.llm | StrOutputParser()
        return chain.invoke({"input": user_prompt})

    def best_report(self, query: str, results: list) -> dict:
        """
        Return ONLY the single best matching + newest report
        from search results.
        """

        user_prompt = f"""
        You are an AI agent that selects the BEST report result.

        Query:
        {query}

        Search results:
        {json.dumps(results, indent=2)}

        Task:
        - Select ONLY ONE result that best matches the query intent
        - Prefer official PDF reports
        - Prefer the newest report (latest year/date in title/content/url)
        - If no true match exists, still return the closest available report

        Output format (STRICT JSON ONLY):

        {{
          "url": "...",
          "title": "...",
          "category": "ir_report / governance_report / other",
          "detected_date": "YYYY-MM-DD or YYYY or null",
          "why_best": "short explanation"
        }}

        Do NOT output anything outside JSON.
        """

        raw = self.run_chatgpt(user_prompt)

        # clean markdown fences
        raw = raw.strip()
        if raw.startswith("```json"):
            raw = raw[7:]
        if raw.startswith("```"):
            raw = raw[3:]
        if raw.endswith("```"):
            raw = raw[:-3]

        raw = raw.strip()

        return json.loads(raw)

validator = SearchReportValidator()

/Users/hoan.hk/Desktop/Works/Stock/venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/hoan.hk/Desktop/Works/Stock/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
file_paths = {
    "hd.eneos.co.jp": {
        "ir_report": [
            "data/hd.eneos.co.jp/00.pdf",
        ],
        "governance_report": [
            "data/hd.eneos.co.jp/system_governance_report.pdf"
        ],
    },

    "mitsubishicorp.com": {
        "ir_report": [
            "data/mitsubishicorp.com/all.pdf",
        ],
        "governance_report": [
            "data/mitsubishicorp.com/governance_report_j.pdf"
        ],
    },

    "lasertec.co.jp": {
        "ir_report": [
            "data/lasertec.co.jp/6920_ir_material_for_fiscal_ym15_192733_00.pdf",
        ],
        "governance_report": [
            "data/lasertec.co.jp/6920_tdnet_2691142_00.pdf"
        ],
    },

    "itochu.co.jp": {
        "ir_report": [
            "data/itochu.co.jp/ja_ir_download___icsFiles_afieldfile_2025_09_05_ar2025J.pdf",
        ],
        "governance_report": [
            "data/itochu.co.jp/ja_files_corporate_governance.pdf"
        ],
    },

    "casio.com": {
        "ir_report": [
            "data/casio.com/content_dam_casio_global_corporate_ir_library_annual_2025_integrated-2025.pdf",
        ],
        "governance_report": [
            "data/casio.com/disclosure_20251224_20251218522473.pdf"
        ],
    },

    "boi.jp": {
        "ir_report": [
            "data/boi.jp/xcontents_AS80485_6a8dfa8d_7071_47c1_ac15_9a4be8a876b8_140120251113500915.pdf",
            "data/boi.jp/xcontents_AS80485_2e4d59dd_a617_4b86_b05d_2fbfda9ec6a2_S100XD6Y.pdf",
        ],
        "governance_report": [
            "data/boi.jp/files_tdnet_140120251119506073.pdf"
        ],
    },

    "mol.co.jp": {
        "ir_report": [
            "data/mol.co.jp/ja_ir_library_integrated_report_main_01_teaserItems2_0_linkList_0_link__J_MOL_20REPORT_2025.pdf"
        ],
        "governance_report": [
            "data/mol.co.jp/sustainability_governance_corporate_policy_pdf_governance-report.pdf"
        ],
    },

    "nintendo.co.jp": {
        "ir_report": [
            "data/nintendo.co.jp/ir_pdf_2025_annual2503e.pdf"
        ],
        "governance_report": [
            "data/nintendo.co.jp/ir_en_management_governance.pdf"
        ],
    },

    "global.toyota": {
        "ir_report": [
            "data/global.toyota/pages_global_toyota_ir_library_annual_2024_001_integrated_en.pdf"
        ],
        "governance_report": [
            "data/global.toyota/files_tdnet_140120250721517384.pdf"
        ],
    },

    "dena.com": {
        "ir_report": [
            "data/dena.com/00_2025_en.pdf"
        ],
        "governance_report": [
            "data/dena.com/files_tdnet_140120251112598427.pdf"
        ],
    },

    "lycorp.co.jp": {
        "ir_report": [
            "data/lycorp.co.jp/integrated_report_FY2024_jp.pdf"
        ],
        "governance_report": [
            "data/lycorp.co.jp/files_tdnet_140120251226527170.pdf"
        ],
    },

    "shinetsu.co.jp": {
        "ir_report": [
            "data/shinetsu.co.jp/統合報告書2025.pdf"
        ],
        "governance_report": [
            "data/shinetsu.co.jp/files_tdnet_140120251223524981.pdf"
        ],
    },

    "bridgestone.co.jp": {
        "ir_report": [
            "data/bridgestone.co.jp/ir2025_single.pdf"
        ],
        "governance_report": [
            "data/bridgestone.co.jp/files_tdnet_140120251031583941.pdf"
        ],
    },

    "capcom.co.jp": {
        "ir_report": [
            "data/capcom.co.jp/ir_english_data_pdf_annual_2025_annual_2025_01.pdf"
        ],
        "governance_report": [
            "data/capcom.co.jp/files_tdnet_140120260106529563.pdf"
        ],
    },

    "fastretailing.com": {
        "ir_report": [
            "data/fastretailing.com/jp_ir_library_pdf_ar2024.pdf"
        ],
        "governance_report": [
            "data/fastretailing.com/jp_about_governance_pdf_governance_report.pdf"
        ],
    },

    "group.softbank": {
        "ir_report": [
            "data/group.softbank/media_Project_sbg_sbg_pdf_ir_financials_annual_reports_annual-report_fy2025_ja.pdf"
        ],
        "governance_report": [
            "data/group.softbank/media_Project_sbg_sbg_pdf_about_corporate_governance_governance_20250704_01_ja.pdf"
        ],
    },

    "daiichisankyo.co.jp": {
        "ir_report": [
            "data/daiichisankyo.co.jp/files_investors_library_annual_report_index_VR2025_ds_vr2025_all_1119.pdf"
        ],
        "governance_report": [
            "data/daiichisankyo.co.jp/files_tdnet_140120251218521839.pdf"
        ],
    },

    "kajima.co.jp": {
        "ir_report": [
            "data/kajima.co.jp/english_sustainability_report_2025_pdf_ir_e_all_2.pdf"
        ],
        "governance_report": [
            "data/kajima.co.jp/files_tdnet_140120250612588145.pdf"
        ],
    },

    "mhi.com": {
        "ir_report": [
            "data/mhi.com/jp_finance_library_annual_pdf_report_2025.pdf"
        ],
        "governance_report": [
            "data/mhi.com/files_tdnet_140120250624597698.pdf"
        ],
    },

    "mitsuifudosan.co.jp": {
        "ir_report": [
            "data/mitsuifudosan.co.jp/corporate_ir_library_integratedreport_pdf_IR2025_ja.pdf"
        ],
        "governance_report": [
            "data/mitsuifudosan.co.jp/files_tdnet_140120250514552438.pdf"
        ],
    },

    "koeitecmo.co.jp": {
        "ir_report": [
            "data/koeitecmo.co.jp/files_tdnet_140120251104586205.pdf"
        ],
        "governance_report": [
            "data/koeitecmo.co.jp/files_tdnet_140120250529573336.pdf"
        ],
    },

    "toei-anim.co.jp": {
        "ir_report": [
            "data/toei-anim.co.jp/en_ir_library_Report_main_00_teaserItems1_0_linkList_0_link_PEROS_20REPORT_202024_en_open.pdf"
        ],
        "governance_report": [
            "data/toei-anim.co.jp/files_tdnet_140120250604582039.pdf"
        ],
    },

    "mufg.jp": {
        "ir_report": [
            "data/mufg.jp/dam_ir_presentation_2025_pdf_slides2509_ja.pdf"
        ],
        "governance_report": [
            "data/mufg.jp/files_tdnet_140120251107591892.pdf"
        ],
    },

    "jfe-holdings.co.jp": {
        "ir_report": [
            "data/jfe-holdings.co.jp/common_pdf_investor_library_group-report_2025_all_A4.pdf"
        ],
        "governance_report": [
            "data/jfe-holdings.co.jp/en_common_pdf_company_info_corporate-governance.pdf"
        ],
    },

    "advantest.com": {
        "ir_report": [
            "data/advantest.com/E_all_IAR2025.pdf"
        ],
        "governance_report": [
            "data/advantest.com/files_tdnet_140120251126509485.pdf"
        ],
    },
}

In [34]:
site = "kajima.co.jp"
query = f"site:{site} filetype:pdf IR report"
headless = False
google_engine = False
results_search = await main(query, headless=headless, google_engine = google_engine)
output = validator.best_report(query, results_search)
print(output)

🔍 Searching: site:kajima.co.jp filetype:pdf IR report
⏳ Waiting for results...
⏳ Waiting for results...
📋 Copying results...
📋 Copying results...
✅ Clipboard captured (4980 chars)
🦆 Parsing in DUCKDUCKGO mode...
✅ Clipboard captured (4980 chars)
🦆 Parsing in DUCKDUCKGO mode...
{'url': 'https://www.kajima.co.jp/english/ir/finance/pdf/20260212-fs.pdf', 'title': '[Summary] Financial Statements for the Third Quarter of FY2025 <under ...', 'category': 'ir_report', 'detected_date': '2026-02-12', 'why_best': "This report is a 'Summary of Financial Statements', which is a key component of an IR report. It is the newest report available, with a document date of February 12, 2026, covering the third quarter of the fiscal year ending March 31, 2026."}
{'url': 'https://www.kajima.co.jp/english/ir/finance/pdf/20260212-fs.pdf', 'title': '[Summary] Financial Statements for the Third Quarter of FY2025 <under ...', 'category': 'ir_report', 'detected_date': '2026-02-12', 'why_best': "This report is a 'S

In [ ]:
from pyserxng.models import SafeSearchLevel, SearchConfig, TimeRange, SearchCategory
from pyserxng import SearXNGClient
from pyserxng.models import InstanceInfo


class SearXNGSearch:
    def __init__(self, instance_url="http://localhost:8888"):
        self.client = SearXNGClient()
        self.instance = InstanceInfo(url=instance_url)

        self.config = SearchConfig(
            page=1,
            safe_search=SafeSearchLevel.STRICT,
            timeout=30,
            engines=["google"],  # 🔥 only google
            categories=[SearchCategory.GENERAL]
        )

        # self.config.time_range = TimeRange.YEAR

    def search(self, query: str):
        print(f"\nQuery: {query}")

        results = self.client.search(
            query,
            instance=self.instance,
            config=self.config
        )

        print(f"Found {len(results.results)} results\n")

        cleaned_results = []

        if results.results:
            for result in results.results:
                cleaned_results.append({
                    "title": str(result.title),
                    "url": str(result.url),
                    "content": str(result.content) if result.content else ""
                })

        else:
            print("No results\n")

        return cleaned_results

In [17]:
import time

all_search_results = {}

searcher = SearXNGSearch()

for domain in file_paths.keys():

    print(f"\n==============================")
    print(f"Processing domain: {domain}")
    print(f"==============================")

    all_search_results[domain] = {}

    for report_type in ["ir report", "governance report"]:

        query = f"site:{domain} filetype:pdf {report_type}"

        try:
            results = searcher.search(query)

            all_search_results[domain][report_type] = results
            time.sleep(5)
        except Exception as e:
            print(f"Error while searching {domain} - {report_type}")
            print(e)
            all_search_results[domain][report_type] = []


Processing domain: hd.eneos.co.jp

Query: site:hd.eneos.co.jp filetype:pdf ir report


2026-02-23 13:38:02,926 - pyserxng.client - INFO - Search completed: 4 results in 1.49s from http://localhost:8888/


Found 4 results


Query: site:hd.eneos.co.jp filetype:pdf governance report

Query: site:hd.eneos.co.jp filetype:pdf governance report


2026-02-23 13:38:09,454 - pyserxng.client - INFO - Search completed: 4 results in 1.50s from http://localhost:8888/


Found 4 results


Processing domain: mitsubishicorp.com

Query: site:mitsubishicorp.com filetype:pdf ir report

Processing domain: mitsubishicorp.com

Query: site:mitsubishicorp.com filetype:pdf ir report


...

Found 10 results


Processing domain: advantest.com

Query: site:advantest.com filetype:pdf ir report

Processing domain: advantest.com

Query: site:advantest.com filetype:pdf ir report


2026-02-23 13:43:20,713 - pyserxng.client - INFO - Search completed: 12 results in 2.16s from http://localhost:8888/


Found 12 results


Query: site:advantest.com filetype:pdf governance report

Query: site:advantest.com filetype:pdf governance report


2026-02-23 13:43:27,123 - pyserxng.client - INFO - Search completed: 4 results in 1.39s from http://localhost:8888/


Found 4 results



In [20]:
validated_results = {}

for domain, reports in all_search_results.items():

    print(f"\n==============================")
    print(f"Validating domain: {domain}")
    print(f"==============================")

    validated_results[domain] = {}

    for report_type, results in reports.items():

        query = f"site:{domain} filetype:pdf {report_type}"
        print(query)

        # if not results:
        #     print(f"No search results for {domain} - {report_type}")
        #     validated_results[domain][report_type] = None
        #     continue

        # try:
        #     output = validator.best_report(query, results)

        #     print(f"\nBest for {domain} - {report_type}:")
        #     print(output)

        #     validated_results[domain][report_type] = output

        # except Exception as e:
        #     print(f"LLM failed for {domain} - {report_type}")
        #     print(e)
        #     validated_results[domain][report_type] = None


Validating domain: hd.eneos.co.jp
site:hd.eneos.co.jp filetype:pdf ir report
site:hd.eneos.co.jp filetype:pdf governance report

Validating domain: mitsubishicorp.com
site:mitsubishicorp.com filetype:pdf ir report
site:mitsubishicorp.com filetype:pdf governance report

Validating domain: lasertec.co.jp
site:lasertec.co.jp filetype:pdf ir report
site:lasertec.co.jp filetype:pdf governance report

Validating domain: itochu.co.jp
site:itochu.co.jp filetype:pdf ir report
site:itochu.co.jp filetype:pdf governance report

Validating domain: casio.com
site:casio.com filetype:pdf ir report
site:casio.com filetype:pdf governance report

Validating domain: boi.jp
site:boi.jp filetype:pdf ir report
site:boi.jp filetype:pdf governance report

Validating domain: mol.co.jp
site:mol.co.jp filetype:pdf ir report
site:mol.co.jp filetype:pdf governance report

Validating domain: nintendo.co.jp
site:nintendo.co.jp filetype:pdf ir report
site:nintendo.co.jp filetype:pdf governance report

Validating doma

In [ ]:
from pyserxng.models import SafeSearchLevel, SearchConfig, TimeRange, SearchCategory
from pyserxng import SearXNGClient
from pyserxng.models import InstanceInfo 

client = SearXNGClient()
local_instance = InstanceInfo(url="http://localhost:8888")

target_site = "kajima.co.jp"

config = SearchConfig(
    page=1,
    language="ja",
    safe_search=SafeSearchLevel.STRICT,
    timeout=30
)

config.time_range = TimeRange.YEAR

query = f"site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 KOEI TECMO HOLDINGS CO., LTD.株式会社コーエーテクモホールディングス"

print(f"Query: {query}")

results = client.search(
    query,
    instance=local_instance,
    config=config
)

print(f"  Found {len(results.results)} results")
print()

if results.results:
    for result in results.results:
        print(result)
        
else:
    print("  No results\n")

Query: site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 KOEI TECMO HOLDINGS CO., LTD.株式会社コーエーテクモホールディングス


2026-02-23 16:21:02,757 - pyserxng.client - INFO - Search completed: 1 results in 2.02s from http://localhost:8888/


  Found 1 results

title='株式会社コーエーテクモホールディングス' url=HttpUrl('https://www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3/20250620/fcjtyw/140120250529573336.pdf') content='2025/06/20 — コーポレートガバナンス.CORPORATEGOVERNANCE.KOEITECMOHOLDINGSCO.,LTD.最終更新日：2025年6月20日. 株式会社コーエーテクモホールディングス. 代表取締役 社長 ...' engine=None category=None score=None thumbnail=None publishedDate=None


In [8]:
from pyserxng.models import SafeSearchLevel, SearchConfig, TimeRange, SearchCategory
from pyserxng import SearXNGClient
from pyserxng.models import InstanceInfo 
import time
import csv

client = SearXNGClient()
local_instance = InstanceInfo(url="http://localhost:8888")

target_site = "kajima.co.jp"

config = SearchConfig(
    page=1,
    language="ja",
    safe_search=SafeSearchLevel.STRICT,
    timeout=30
)

# config.time_range = TimeRange.YEAR

In [21]:
import os
from langchain_openai import ChatOpenAI

# Configure ProxyPal for ChatGPT
os.environ["OPENAI_API_KEY"] = "proxypal-local"
os.environ["OPENAI_API_BASE"] = "http://localhost:8317/v1"

import os
import json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

class SearchReportValidator:
    def __init__(self, model_name="gemini-2.5-flash"):
        self.model_name = model_name
        self.llm = ChatOpenAI(model=model_name, temperature=0.0)

    def run_chatgpt(self, user_prompt: str) -> str:
        prompt = ChatPromptTemplate.from_messages([
            ("user", "{input}")
        ])
        chain = prompt | self.llm | StrOutputParser()
        return chain.invoke({"input": user_prompt})

    def best_report(self, query: str, results: list) -> dict:
        """
        Return ONLY the single best matching + newest report
        from search results.
        """

        user_prompt = f"""
        You are an AI agent that selects the BEST report result.

        Query:
        {query}

        Search results:
        {json.dumps(results, indent=2)}

        Task:
        - Select ONLY ONE result that best matches the query intent
        - Prefer official PDF reports
        - Prefer the newest report (latest year/date in title/content/url)
        - If no true match exists, still return the closest available report

        Output format (STRICT JSON ONLY):

        {{
          "url": "...",
          "title": "...",
          "category": "ir_report / governance_report / other",
          "detected_date": "YYYY-MM-DD or YYYY or null",
          "why_best": "short explanation"
        }}

        Do NOT output anything outside JSON.
        """

        raw = self.run_chatgpt(user_prompt)

        # clean markdown fences
        raw = raw.strip()
        if raw.startswith("```json"):
            raw = raw[7:]
        if raw.startswith("```"):
            raw = raw[3:]
        if raw.endswith("```"):
            raw = raw[:-3]

        raw = raw.strip()

        return json.loads(raw)

validator = SearchReportValidator()

In [ ]:

companies = [
    "TOEI ANIMATION CO.,LTD. 東映アニメーション株式会社",
    "Mitsui O.S.K. Lines, Ltd. 株式会社商船三井",
    "TOYOTA MOTOR CORPORATION トヨタ自動車株式会社",
    "JFE Holdings, Inc. ＪＦＥホールディングス株式会社",
    "Mitsui Fudosan Co., Ltd. 三井不動産株式会社",
    "KOEI TECMO HOLDINGS CO., LTD. 株式会社コーエーテクモホールディングス",
    "DeNA Co., Ltd. 株式会社ディー・エヌ・エー",
    "Lasertec Corporation レーザーテック株式会社",
    "Shin-Etsu Chemical Co., Ltd. 信越化学工業株式会社",
    "CAPCOM CO., LTD. 株式会社カプコン",
    "BRIDGESTONE CORPORATION 株式会社ブリヂストン",
    "DAIICHI SANKYO COMPANY, LIMITED 第一三共株式会社",
    "KAJIMA CORPORATION 鹿島建設株式会社",
    "Nintendo Co., Ltd. 任天堂株式会社",
    "Mitsubishi Heavy Industries, Ltd. 三菱重工業株式会社",
    "ITOCHU Corporation 伊藤忠商事株式会社",
    "CASIO COMPUTER CO., LTD. カシオ計算機株式会社",
    "Bank of Innovation, Inc. 株式会社バンク・オブ・イノベーション",
    "Mitsubishi UFJ Financial Group, Inc. 株式会社三菱ＵＦＪフィナンシャル・グループ",
    "ADVANTEST CORPORATION 株式会社アドバンテスト",
    "FAST RETAILING CO., LTD. 株式会社ファーストリテイリング",
    "SoftBank Group Corp. ソフトバンクグループ株式会社",
    "ENEOS Holdings, Inc. ＥＮＥＯＳホールディングス株式会社",
    "Mitsubishi Corporation 三菱商事株式会社",
    "LY Corporation ＬＩＮＥヤフー株式会社"
]

output_file = "nikkei_governance_best_results.csv"

# ========================
# CREATE CSV HEADER
# ========================

with open(output_file, mode="w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow([
        "company_name",
        "title",
        "url",
        "category",
        "detected_date",
        "why_best"
    ])

# ========================
# MAIN LOOP
# ========================

for name in companies:

    print("\n" + "="*100)
    print("Company:", name)
    print("="*100)

    query = f"""
    site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 {name} filetype:pdf
    """

    print("Query:", query.strip())

    results = client.search(
        query,
        instance=local_instance,
        config=config
    )

    search_results = []

    if results.results:
        for r in results.results:
            search_results.append({
            "title": str(getattr(r, "title", "")),
            "content": str(getattr(r, "content", "")),
            "url": str(getattr(r, "url", ""))
        })

        best = validator.best_report(query, search_results)
        print(best)

        with open(output_file, mode="a", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            writer.writerow([
                name,
                best.get("title"),
                best.get("url"),
                best.get("category"),
                best.get("detected_date"),
                best.get("why_best")
            ])

        print("✅ Saved best report:", best.get("title"))


    else:
        print("⚠ No search results found")

    time.sleep(10)

print("\nDONE. File saved:", output_file)


Company: TOEI ANIMATION CO.,LTD. 東映アニメーション株式会社
Query: site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 TOEI ANIMATION CO.,LTD. 東映アニメーション株式会社 filetype:pdf


2026-02-23 17:50:59,152 - pyserxng.client - INFO - Search completed: 6 results in 2.41s from http://localhost:8888/


{'url': 'https://www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3/20250707/fcwvnb/140120250604582039.pdf', 'title': 'コーポレート・ガバナンスに関する報告書 2025/ ...', 'category': 'governance_report', 'detected_date': '2025-07-07', 'why_best': "This report is for 'TOEI ANIMATION CO.,LTD.' and is a 'CORPORATE GOVERNANCE' report, matching the query's specific company and document type. It also has the latest '最終更新日' (last updated date) of 2025-07-07 among all relevant results."}
✅ Saved best report: コーポレート・ガバナンスに関する報告書 2025/ ...

Company: Mitsui O.S.K. Lines, Ltd. 株式会社商船三井
Query: site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 Mitsui O.S.K. Lines, Ltd. 株式会社商船三井 filetype:pdf

Company: Mitsui O.S.K. Lines, Ltd. 株式会社商船三井
Query: site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 Mitsui O.S.K. Lines, Ltd. 株式会社商船三井 filetype:pdf


2026-02-23 17:51:19,304 - pyserxng.client - INFO - Search completed: 6 results in 2.70s from http://localhost:8888/


{'url': 'https://www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3/20250731/fffmv0/140120250717516348.pdf', 'title': 'コーポレートガバナンス', 'category': 'governance_report', 'detected_date': '2025-07-31', 'why_best': "This result is an official PDF report explicitly titled 'Corporate Governance' for 'Mitsui O.S.K. Lines, Ltd.' and contains the '最終更新日' (last updated date) of 2025-07-31, which is the newest date among all relevant search results, perfectly matching all aspects of the query."}
✅ Saved best report: コーポレートガバナンス

Company: TOYOTA MOTOR CORPORATION トヨタ自動車株式会社
Query: site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 TOYOTA MOTOR CORPORATION トヨタ自動車株式会社 filetype:pdf

Company: TOYOTA MOTOR CORPORATION トヨタ自動車株式会社
Query: site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 TOYOTA MOTOR CORPORATION トヨタ自動車株式会社 filetype:pdf


...

2026-02-23 17:55:16,777 - pyserxng.client - INFO - Search completed: 5 results in 2.41s from http://localhost:8888/


{'url': 'https://www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3/20260203/frcwsi/140120260202544642.pdf', 'title': 'コーポレート・ガバナンス報告書', 'category': 'governance_report', 'detected_date': '2026-02-03', 'why_best': 'This report is a Corporate Governance report for KAJIMA CORPORATION, is in PDF format from the specified domain, and has the latest update date (2026-02-03) among all the search results.'}
✅ Saved best report: コーポレート・ガバナンス報告書

Company: Nintendo Co., Ltd. 任天堂株式会社
Query: site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 Nintendo Co., Ltd. 任天堂株式会社 filetype:pdf

Company: Nintendo Co., Ltd. 任天堂株式会社
Query: site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 Nintendo Co., Ltd. 任天堂株式会社 filetype:pdf


2026-02-23 17:55:32,277 - pyserxng.client - INFO - Search completed: 10 results in 1.89s from http://localhost:8888/


{'url': 'https://www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3/20240628_archive/eqq0qv/140120240528510503.pdf', 'title': '任天堂株式会社', 'category': 'governance_report', 'detected_date': '2024-06-28', 'why_best': "This result is a Corporate Governance report specifically for Nintendo Co., Ltd. (任天堂株式会社) and has the latest '最終更新日' (last updated date) of 2024年6月28日 (June 28, 2024) among all relevant results."}
✅ Saved best report: 任天堂株式会社

Company: Mitsubishi Heavy Industries, Ltd. 三菱重工業株式会社
Query: site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 Mitsubishi Heavy Industries, Ltd. 三菱重工業株式会社 filetype:pdf

Company: Mitsubishi Heavy Industries, Ltd. 三菱重工業株式会社
Query: site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 Mitsubishi Heavy Industries, Ltd. 三菱重工業株式会社 filetype:pdf


2026-02-23 17:55:54,122 - pyserxng.client - INFO - Search completed: 10 results in 2.45s from http://localhost:8888/
